# 1 Initialize the Database

All the code related to data management is in the `EnvironmentData` class. This makes life easier - for example: we can send the CatsUserID once and it becomes a class property. Then, when we call other operations we don't have to send this information again.

When you create a new instance of `EnvironmentData` and there is no database, it will pull historical data and initialize the database. 

In [ ]:
# Clear prior data. 
import os, sys, shutil

# Add parent directory to Python path to import EnvironmentData.
sys.path.append(os.path.dirname(os.getcwd()))

# Get the EnvironmentData class.
from EnvironmentData import EnvironmentData 

# The project adds to existing data so we need to clear that data to get a solid test from scratch.
if os.path.exists('../data'):
    shutil.rmtree('../data')
    os.makedirs('../data')

# Initialize EnvironmentData. This will run the historical data pull.
envdt = EnvironmentData(
    days_back = 365 * 2,
    out_of_scope = ['-80', 'Cryo tank', 'Water'],
    coris_enabled = False,
    licor_enabled = False,
    conserv_enabled = True, 
    testing = True,
    # Since we are running from the experiments/ folder, we need to tell the class to use the parent directory as home.
    home_directory = ".."
)

DEBUG: Enabled data sources: ['Conserv']


Detailed information is saved in the log:

In [2]:
# Detailed info is saved in the log.
with open('../data/EnvironmentData.log', 'r') as file:
    for line in file.read().splitlines()[:10]:
        print(line)

2025-11-15 09:31:27,537 - EnvironmentData - INFO - Enabled data sources: ['Coris', 'LI-COR']
2025-11-15 09:31:27,538 - EnvironmentData - INFO - API call: https://cats.corismonitoring.com/api/cats/user/?ApiKey=XXXX&CatsUserID=XXXX
2025-11-15 09:31:29,246 - EnvironmentData - INFO - API call: https://cats.corismonitoring.com/api/sensor/historical/?ApiKey=XXXX&SensorID=21373&ReadingType=SensorReadingF&StartUTC=1700152287&EndUTC=1763224287&MinReadingSpacing=600&RequestedOutputFormat=raw
2025-11-15 09:31:35,039 - EnvironmentData - INFO - API call: https://cats.corismonitoring.com/api/sensor/historical/?ApiKey=XXXX&SensorID=21375&ReadingType=SensorReadingF&StartUTC=1700152287&EndUTC=1763224287&MinReadingSpacing=600&RequestedOutputFormat=raw
2025-11-15 09:31:41,913 - EnvironmentData - INFO - API call: https://cats.corismonitoring.com/api/sensor/historical/?ApiKey=XXXX&SensorID=21377&ReadingType=SensorReadingF&StartUTC=1700152287&EndUTC=1763224287&MinReadingSpacing=600&RequestedOutputFormat=raw

This saves our intermediate data to `data/sensor_readings.parquet`. 

Initially, we leave the data mostly as-is. We'll clean, add formatted dates, consolidate readings from the same device, etc. when moving to analytical steps, this preserves the source data so we can always change our mind later about how we decide to view it. 

However, at this point we are taking care to standardize the data format between different API sources. 

There are just a few columns because this is only historical data. We'll bring in current data shortly, and that will add more columns. 

In [3]:
import polars
polars.read_parquet('../data/sensor_readings.parquet').filter(polars.col("Source") == "Coris").head()

SensorReadingUTC,Source,DeviceID,DeviceName,SensorID,SensorName,SensorType,SensorReadingF,SensorReadingRh
i64,str,str,str,str,str,str,f32,f32
1700152287,"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21373""","""PYPM__0100104SET____ Temp YPM …","""Temperature""",70.790001,null
1700152887,"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21373""","""PYPM__0100104SET____ Temp YPM …","""Temperature""",70.809998,null
1700153487,"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21373""","""PYPM__0100104SET____ Temp YPM …","""Temperature""",70.790001,null
1700154087,"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21373""","""PYPM__0100104SET____ Temp YPM …","""Temperature""",70.809998,null
1700154687,"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21373""","""PYPM__0100104SET____ Temp YPM …","""Temperature""",70.790001,null


In [4]:
polars.read_parquet('../data/sensor_readings.parquet').filter(polars.col("Source") == "LI-COR").head()

SensorReadingUTC,Source,DeviceID,DeviceName,SensorID,SensorName,SensorType,SensorReadingF,SensorReadingRh
i64,str,str,str,str,str,str,f32,f32
1734112500,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179175-2""",null,"""RH""",null,11.764706
1734112800,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179175-2""",null,"""RH""",null,11.567864
1734113100,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179175-2""",null,"""RH""",null,14.859236
1734113400,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179175-2""",null,"""RH""",null,15.115587
1734113700,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179175-2""",null,"""RH""",null,14.169528


# 2 Get Current Readings

Now we can start gathering and appending readings. There is a function `get_current_readings` that is run throughout the day, every 10 minutes for example. This function creates a parquet file at `data/new-readings` with the UTC as a filename. At the end of the day, all these readings will be consolidated into the database. 

Here is a sample of the readings:

In [5]:
envdt.get_current_readings()

# Data is read into new-readings folder for consolidation at the end of the day.
import os
filename = os.listdir('../data/new-readings')[0]
print(filename)
polars.read_parquet('../data/new-readings/' + filename).sample(5)

1763224329.parquet


SensorReadingUTC,Source,DeviceID,DeviceName,SensorID,SensorName,SensorType,SensorReadingF,SensorReadingRh,QueryUTC
i64,str,str,str,str,str,str,f32,f32,i32
1763224332,"""LI-COR""","""licor:10740550""","""ICSC__010C149_______""","""licor:10740550-10740550-1""","""ICSC__010C149________Temperatu…","""Temperature""",65.829819,null,1763224331
1763224227,"""Coris""","""coris:12169""","""Peabody TH-L Diorama Room 304 …","""coris:21376""","""PYPM__0300302SET____ RH YPM 30…","""Humidity""",null,50.07,1763224329
1763224332,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179174-2""","""RX Station 1_RH""","""RH""",null,37.964447,1763224331
1763224332,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179174-1""","""RX Station 1_Temperature""","""Temperature""",70.142036,null,1763224331
1763224332,"""LI-COR""","""licor:22202141""","""RX Station 2""","""licor:22202141-22179175-1""","""RX Station 2_Temperature""","""Temperature""",70.103424,null,1763224331


# 3 Consolidate Readings

At the end of the day, new readings will be consolidated into the table. At the same time, the analytical tables will be generated. 

Analytical tables include:

* `device_readings.parquet`: Sensor readings reorganized to one row per Device and UTC, with measurements across columns vs measurements across rows.* 
* `sensors.parquet`: Information about the unique sensors. Includes information extracted from SensorName. Join this to Sensors during analysis to enhance with Building, Room, Direction, etc.
* `devices.parquet`: Information about unique devices. Includes information extracted from SensorName. 
* `utcs.parquet`: Information related to the UTC times in various datasets. Join to Sensors or Devices to enhance with Date, Time, Year, Hour, Weekday, etc.
* `sensor_readings_daily.parquet`: Example of sensor readings summarized to the daily level which reduces row count by 99.3% for even faster queries.
* `device_readings_daily.parquet`: Example of device readings summarized to the daily level which reduces row count by 99.3% for even faster queries. 

We fully re-generate analytical tables during each consolidation. The data is small enough that this is a fairly quick process, so re-running it in full each time will make it easy to ensure consistency as we expand and change the project. 

In [6]:
# To consolidate these into the database, run consolidate_readings.
envdt.consolidate_readings()

# New-readings files are gone now.
# They get deleted each day to confirm that they have been loaded into the database and prepare for the next consolidation.
print(os.listdir('../data/new-readings'))

C:\Users\super\Documents\arbaiza-consulting\environmental-sensor-poc\EnvironmentData.py:231: UserWarning: consolidate_readings validation errors : Count of sensors missing from historical data: 38.

  warnings.warn(msg)


[]


**^^ We want this to be empty** since we have consolidated new readings into the historical data. 

Once we are done working with data intake/processing, we close the class to release the file lock on the log file.

In [7]:
# When done, close the connection to the logs. 
envdt.close()

Let's look at the data we have now:

In [8]:
# Sensor Readings
# The first rows will be missing the extra fields like HexGatewayMac, etc.
#   I am pulling in some extra fields like DeviceID and DeviceName so we have that by historical. 
#   But some don't make sense to  backfill so they'll be null.
sensor_readings = polars.read_parquet('../data/sensor_readings.parquet')
sensor_readings.head()

SensorReadingUTC,Source,DeviceID,DeviceName,SensorID,SensorName,SensorType,SensorReadingF,SensorReadingRh,QueryUTC,SensorReadingUTC_SecondsFromPrior
i64,str,str,str,str,str,str,f32,f32,i32,i64
1700152287,"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21373""","""PYPM__0100104SET____ Temp YPM …","""Temperature""",70.790001,null,null,null
1700152887,"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21373""","""PYPM__0100104SET____ Temp YPM …","""Temperature""",70.809998,null,null,600
1700153487,"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21373""","""PYPM__0100104SET____ Temp YPM …","""Temperature""",70.790001,null,null,600
1700154087,"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21373""","""PYPM__0100104SET____ Temp YPM …","""Temperature""",70.809998,null,null,600
1700154687,"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21373""","""PYPM__0100104SET____ Temp YPM …","""Temperature""",70.790001,null,null,600


In [9]:
# Recent rows will have the full data, aside from nulls due to a sensor not providing a reading type.
sensor_readings.tail()

SensorReadingUTC,Source,DeviceID,DeviceName,SensorID,SensorName,SensorType,SensorReadingF,SensorReadingRh,QueryUTC,SensorReadingUTC_SecondsFromPrior
i64,str,str,str,str,str,str,f32,f32,i32,i64
1763217900,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179175-2""",null,"""RH""",null,40.67445,null,900
1763218800,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179175-2""",null,"""RH""",null,48.067444,null,900
1763219700,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179175-2""",null,"""RH""",null,42.931259,null,900
1763220600,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179175-2""",null,"""RH""",null,36.5774,null,900
1763224332,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179175-2""","""RX Station 1_RH""","""RH""",null,36.5774,1763224331,3732


In [10]:
# Device Readings.
device_readings = polars.read_parquet('../data/device_readings.parquet')
device_readings.head()

Source,DeviceID,DeviceName,Sensors,SensorNames,SensorTypes,SensorReadingUTC,QueryUTC,SensorReadingF,SensorReadingRh
str,str,str,str,str,str,i64,i32,f32,f32
"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21374, coris:21373""","""PYPM__0100104SET____ RH YPM 10…","""Humidity, Temperature""",1700152287,null,70.790001,null
"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21374, coris:21373""","""PYPM__0100104SET____ RH YPM 10…","""Humidity, Temperature""",1700152887,null,70.809998,null
"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21374, coris:21373""","""PYPM__0100104SET____ RH YPM 10…","""Humidity, Temperature""",1700153487,null,70.790001,null
"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21374, coris:21373""","""PYPM__0100104SET____ RH YPM 10…","""Humidity, Temperature""",1700154087,null,70.809998,null
"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21374, coris:21373""","""PYPM__0100104SET____ RH YPM 10…","""Humidity, Temperature""",1700154687,null,70.790001,null


In [11]:
# Sensors
sensors = polars.read_parquet('../data/sensors.parquet')
sensors.head()

Source,SensorID,SensorName,SensorType,DeviceID,DeviceSerialFromName,BuildingID,Building,Room,CardinalDirection
str,str,str,str,str,str,str,str,str,str
"""LI-COR""","""licor:10740550-10740550-2""","""ICSC__010C149________RH""","""RH""","""licor:10740550""","""RH""","""FLOATER""","""Unknown""","""FLOATER""",null
"""LI-COR""","""licor:10740550-10740550-1""","""ICSC__010C149________Temperatu…","""Temperature""","""licor:10740550""","""Temperature""","""FLOATER""","""Unknown""","""FLOATER""",null
"""Coris""","""coris:21378""","""RH KGL 21_D0B2""","""Humidity""","""coris:12167""","""D0B2""","""KGL""","""Kline Geology Laboratory""","""21""","""Not Indicated"""
"""Coris""","""coris:21377""","""Temp KGL 21_D0B2""","""Temperature""","""coris:12167""","""D0B2""","""KGL""","""Kline Geology Laboratory""","""21""","""Not Indicated"""
"""LI-COR""","""licor:22202142-22179174-2""","""RX Station 1_RH""","""RH""","""licor:22202142""","""RH""","""Station""","""Unknown""","""1""","""Not Indicated"""


In [12]:
# Devices. 
devices = polars.read_parquet('../data/devices.parquet')
devices.head()

Source,DeviceID,DeviceName,Sensors,SensorNames,SensorTypes,DeviceSerialFromName,BuildingID,Building,Room,CardinalDirection
str,str,str,str,str,str,str,str,str,str,str
"""LI-COR""","""licor:10740550""","""ICSC__010C149_______""","""licor:10740550-10740550-2, lic…","""ICSC__010C149________RH, ICSC_…","""RH, Temperature""","""RH""","""FLOATER""","""Unknown""","""FLOATER""",null
"""Coris""","""coris:12167""","""Peabody TH-L Upper Great Hall …","""coris:21378, coris:21377""","""RH KGL 21_D0B2, Temp KGL 21_D0…","""Humidity, Temperature""","""D0B2""","""KGL""","""Kline Geology Laboratory""","""21""","""Not Indicated"""
"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179174-2, lic…","""RX Station 1_RH, RX Station 1_…","""RH, RH, Temperature, Temperatu…","""Temperature""","""Station""","""Unknown""","""1""","""Not Indicated"""
"""LI-COR""","""licor:22202141""","""RX Station 2""","""licor:22202141-22179175-2, lic…","""RX Station 2_RH, RX Station 2_…","""RH, Temperature""","""RH""","""Station""","""Unknown""","""2""","""Not Indicated"""
"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21374, coris:21373""","""RH YPM 104_D444 , Temp YPM 104…","""Humidity, Temperature""","""D444""","""YPM""","""Yale Peabody Museum""","""104""","""Not Indicated"""


In [13]:
# UTC Date/Time Info
utcs = polars.read_parquet('../data/utcs.parquet').head()
utcs.head()

UTC,datetime_utc,datetime_est,date,time,year,month,day_of_week,day_of_week_monday1_sunday7,hour_24,hour_12,am_pm
i64,datetime[μs],"datetime[μs, America/New_York]",date,time,i32,i8,str,i8,i8,i8,str
1753743360,2025-07-28 16:56:00,2025-07-28 12:56:00 EDT,2025-07-28,12:56:00,2025,7,"""Monday""",1,12,0,"""PM"""
1753481220,2025-07-25 16:07:00,2025-07-25 12:07:00 EDT,2025-07-25,12:07:00,2025,7,"""Friday""",5,12,0,"""PM"""
1715732487,2024-05-14 18:21:27,2024-05-14 14:21:27 EDT,2024-05-14,14:21:27,2024,5,"""Tuesday""",2,14,2,"""PM"""
1735393287,2024-12-28 06:41:27,2024-12-28 01:41:27 EST,2024-12-28,01:41:27,2024,12,"""Saturday""",6,1,1,"""AM"""
1753219080,2025-07-22 15:18:00,2025-07-22 11:18:00 EDT,2025-07-22,11:18:00,2025,7,"""Tuesday""",2,11,11,"""AM"""


In [14]:
# Daily Sensor Readings
# Averages are calculated by summing the "sum" and "row_count" columns and dividing to get the average. 
sensor_readings_daily = polars.read_parquet('../data/sensor_readings_daily.parquet')
sensor_readings_daily.head()

Source,date,SensorID,row_count,SensorReadingF_sum,SensorReadingRh_sum,SensorReadingF_min,SensorReadingRh_min,SensorReadingF_max,SensorReadingRh_max
str,date,str,u32,f32,f32,f32,f32,f32,f32
"""Coris""",2023-11-16,"""coris:21373""",117,8230.749023,0.0,69.800003,null,70.919998,null
"""Coris""",2023-11-16,"""coris:21374""",117,0.0,5728.020508,null,47.709999,null,49.77
"""Coris""",2023-11-16,"""coris:21375""",117,7981.562988,0.0,68.07,null,68.360001,null
"""Coris""",2023-11-16,"""coris:21376""",117,0.0,5749.491211,null,47.919998,null,51.080002
"""Coris""",2023-11-16,"""coris:21377""",117,8523.317383,0.0,72.25,null,73.580002,null


In [15]:
# Daily Device Readings
# Averages are calculated by summing the "sum" and "row_count" columns and dividing to get the average. 
device_readings_daily = polars.read_parquet('../data/device_readings_daily.parquet')
device_readings_daily.head()

Source,date,DeviceID,row_count,SensorReadingF_sum,SensorReadingRh_sum,SensorReadingF_min,SensorReadingRh_min,SensorReadingF_max,SensorReadingRh_max
str,date,str,u32,f32,f32,f32,f32,f32,f32
"""Coris""",2023-11-16,"""coris:12162""",234,8230.749023,5728.020508,69.800003,47.709999,70.919998,49.77
"""Coris""",2023-11-16,"""coris:12167""",234,8523.317383,4505.799805,72.25,36.630001,73.580002,39.950001
"""Coris""",2023-11-16,"""coris:12169""",234,7981.562988,5749.491211,68.07,47.919998,68.360001,51.080002
"""Coris""",2023-11-17,"""coris:12162""",288,10010.748047,7754.789551,68.769997,47.919998,70.449997,57.389999
"""Coris""",2023-11-17,"""coris:12167""",288,10465.308594,6665.239746,71.559998,37.580002,73.400002,51.360001


In [17]:
# Differentiate historical vs. cron readings by filtering on QueryUTC = NULL.
import duckdb
duckdb.sql("""
    SELECT *
    FROM read_parquet('../data/device_readings.parquet') 
    WHERE QueryUTC is null
    LIMIT 5
""").to_df()

,Source,DeviceID,DeviceName,Sensors,SensorNames,SensorTypes,SensorReadingUTC,QueryUTC,SensorReadingF,SensorReadingRh
0,Coris,coris:12162,Peabody TH-L Mammal Hall D444,"coris:21374, coris:21373","PYPM__0100104SET____ RH YPM 104_D444 , PYPM__0...","Humidity, Temperature",1700152287,<NA>,70.790001,NaN
1,Coris,coris:12162,Peabody TH-L Mammal Hall D444,"coris:21374, coris:21373","PYPM__0100104SET____ RH YPM 104_D444 , PYPM__0...","Humidity, Temperature",1700152887,<NA>,70.809998,NaN
2,Coris,coris:12162,Peabody TH-L Mammal Hall D444,"coris:21374, coris:21373","PYPM__0100104SET____ RH YPM 104_D444 , PYPM__0...","Humidity, Temperature",1700153487,<NA>,70.790001,NaN
3,Coris,coris:12162,Peabody TH-L Mammal Hall D444,"coris:21374, coris:21373","PYPM__0100104SET____ RH YPM 104_D444 , PYPM__0...","Humidity, Temperature",1700154087,<NA>,70.809998,NaN
4,Coris,coris:12162,Peabody TH-L Mammal Hall D444,"coris:21374, coris:21373","PYPM__0100104SET____ RH YPM 104_D444 , PYPM__0...","Humidity, Temperature",1700154687,<NA>,70.790001,NaN


Now you are ready to move onto analysis to get human-readable results (not indexed by UTC timestamps). See 2-examples-analysis.ipynb.